# 18.6 Synthetic data generation

Synthetic data is useful when it preserves the learning question, not merely when it creates more rows. Synthetic generation sits after EDA, sampling design, and validation because it must imitate useful structure without copying private accidents. Save a copy to Drive to edit.

In [ ]:

import hashlib
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(18)


def dataquality_ladder():
    """Breast Cancer ladder with progressively degraded data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def stable_split(X, y):
    return train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=18,
        stratify=y,
    )


def fit_logistic_accuracy(x_train, y_train, x_test, y_test):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_train_scaled, y_train)
    pred = clf.predict(x_test_scaled)
    return float(accuracy_score(y_test, pred))


def preview_ladder(rungs):
    for name, X, y in rungs:
        counts = np.bincount(y.astype(int))
        sample = np.round(X[:2, :4], 3)
        print(name)
        print("  shape", X.shape, "class_counts", counts.tolist())
        print("  first_two_by_four")
        print(sample)


## The concept, built once (D1)

A generator samples $\tilde x\sim q(x\mid y)$, then we audit means, spread, class balance, nearest-neighbor distance, and downstream utility. The lesson's checks are: real mean $2.500$, real standard deviation $1.118$, synthetic distances $\{0.05,0.8,1.0\}$, and minority repair $5+35=40$ positives.

In [ ]:

def lesson_synthetic_checks():
    real_values = np.array([1.0, 2.0, 3.0, 4.0])
    synthetic_values = np.array([1.2, 1.9, 3.1, 3.8])
    real_points = np.array([0.0, 1.0, 3.0, 6.0]).reshape(-1, 1)
    synthetic_points = np.array([0.05, 2.2, 5.0]).reshape(-1, 1)
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(real_points)
    distances = nn.kneighbors(synthetic_points, return_distance=True)[0].ravel()
    checks = {
        "real_mean": float(real_values.mean()),
        "synthetic_mean": float(synthetic_values.mean()),
        "real_std": float(real_values.std()),
        "distances": distances,
        "balanced_positive_count": 5 + 35,
        "negative_count": 40,
    }
    return checks


checks = lesson_synthetic_checks()
print(checks)
assert round(checks["real_mean"], 3) == 2.500
assert round(checks["synthetic_mean"], 3) == 2.500
assert round(checks["real_std"], 3) == 1.118
assert np.allclose(np.round(checks["distances"], 2), [0.05, 0.8, 1.0])
assert checks["balanced_positive_count"] == checks["negative_count"] == 40


Now package the audit with class-conditional Gaussian generation. Synthetic rows are created in standardized space, filtered when they are too close to real rows, and evaluated by the same logistic model.

In [ ]:

def generate_check_train(X, y, target_minority=None, noise_scale=0.35, min_distance=0.08, filter_close=True):
    x_train, x_test, y_train, y_test = stable_split(X, y)
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    counts = np.bincount(y_train.astype(int))
    minority = int(np.argmin(counts))
    majority_count = int(np.max(counts))
    wanted = majority_count if target_minority is None else target_minority
    needed = max(0, wanted - int(counts[minority]))
    rng = np.random.default_rng(186)
    class_rows = x_train_scaled[y_train == minority]
    center = class_rows.mean(axis=0)
    scale = class_rows.std(axis=0) + 1e-6
    synthetic = rng.normal(center, noise_scale * scale, size=(needed, x_train_scaled.shape[1]))

    if needed > 0:
        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(x_train_scaled)
        distances = nn.kneighbors(synthetic, return_distance=True)[0].ravel()
        if filter_close:
            keep = distances >= min_distance
            synthetic = synthetic[keep]
            distances = distances[keep]
    else:
        distances = np.array([np.inf])

    if len(synthetic) > 0:
        y_synthetic = np.full(len(synthetic), minority)
        x_aug = np.vstack([x_train_scaled, synthetic])
        y_aug = np.concatenate([y_train, y_synthetic])
    else:
        x_aug = x_train_scaled
        y_aug = y_train

    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_aug, y_aug)
    pred = clf.predict(x_test_scaled)
    accuracy = accuracy_score(y_test, pred)
    warning = bool(np.min(distances) < min_distance)
    return {
        "accuracy": float(accuracy),
        "synthetic_rows": int(len(synthetic)),
        "min_distance": float(np.min(distances)),
        "privacy_warning": warning,
        "class_counts_after": np.bincount(y_aug.astype(int)).tolist(),
        "artifact": synthetic[:, :2] if len(synthetic) else np.empty((0, 2)),
    }


## The dataset ladder

The same Breast Cancer dataset is progressively corrupted, so changes in the curve come from data quality rather than model choice.

In [ ]:

rungs = dataquality_ladder()
preview_ladder(rungs)


## Run the same method across D1-D5

In [ ]:

results = []
for name, X, y in rungs:
    result = generate_check_train(X, y)
    row = {
        "rung": name,
        "accuracy": result["accuracy"],
        "synthetic_rows": result["synthetic_rows"],
        "min_distance": result["min_distance"],
        "privacy_warning": result["privacy_warning"],
        "artifact": result["artifact"],
    }
    results.append(row)
    print(f"{name:28s} acc={row['accuracy']:.3f} synthetic={row['synthetic_rows']:3d} min_nn={row['min_distance']:.3f} warning={row['privacy_warning']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, results):
    art = row["artifact"]
    if len(art) > 0:
        ax.scatter(art[:, 0], art[:, 1], s=10, alpha=0.7)
    ax.set_title(row["rung"].split()[0])
    ax.set_xlabel("synthetic f0")
    ax.set_ylabel("synthetic f1")
fig.suptitle("Synthetic rows generated per rung")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), [r["accuracy"] for r in results], marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0.6, 1.0)
plt.ylabel("accuracy")
plt.title("Utility vs data quality")
plt.grid(True)
plt.show()


## Pitfall on D5: memorization disguised as generation

Copied-neighbor synthetic rows can boost row count while failing privacy review. The fix filters rows below a nearest-neighbor threshold and regenerates farther class-conditional samples.

In [ ]:

name, X, y = rungs[-1]
x_train, x_test, y_train, y_test = stable_split(X, y)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
rng = np.random.default_rng(618)
copied = x_train_scaled[:40] + rng.normal(0.0, 0.005, size=x_train_scaled[:40].shape)
nn = NearestNeighbors(n_neighbors=1)
nn.fit(x_train_scaled)
copy_distances = nn.kneighbors(copied, return_distance=True)[0].ravel()
wrong_min = float(copy_distances.min())
fixed = generate_check_train(X, y, noise_scale=0.6, min_distance=0.08, filter_close=True)
print("wrong copied min distance", round(wrong_min, 4))
print("fixed generated min distance", round(fixed["min_distance"], 4))
print("fixed accuracy", round(fixed["accuracy"], 3))
assert wrong_min < 0.10
assert fixed["min_distance"] >= 0.08



## Evaluate it + Practice

- Compare the reported accuracy to a no-skill majority-class baseline for every rung.
- Sanity-check that the key data artifact changes in the intended direction before trusting the metric.
- Ablate the key data-centric step, such as filtering, augmentation, validation, version selection, or valuation-guided dropping.
- Watch for failure signals: gains only on corrupted validation data, impossible feature values, leakage, or unstable row rankings.
- Re-run with a new seed and require the conclusion, not every decimal, to remain stable.

Practice 1: change the corruption or repair strength and re-run the D1 to D5 curve.


Practice 2: add one slice-level diagnostic for the hardest rung.

Practice 3: replace accuracy with balanced accuracy or F1 and compare the story.